In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [2]:
!pip install -q transformers accelerate datasets

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
messages = [
    {"role": "user", "content": "What is 12 + 7?"}
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is 12 + 7?<|im_end|>
<|im_start|>assistant



In [5]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
test_set = gsm8k["test"]
print(test_set[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [6]:
FEWSHOT = """Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
A: In April, Natalia sold 48 clips. In May, she sold half as many, so 48 / 2 = 24 clips. Altogether she sold 48 + 24 = 72 clips. The answer is 72.

Q: A store sells notebooks for $3 each. If Kylar buys 16 notebooks and gets a 20% discount on the total price, how much does he pay?
A: The full price for 16 notebooks is 16 * $3 = $48. A 20% discount on $48 is 0.20 * 48 = $9.60. So Kylar pays 48 - 9.60 = $38.40. The answer is 38.4.

"""

INSTRUCTION = "End your response with 'The answer is X.' where X is a number, and do not write anything after that."

def build_prompt(question):
    full_question = f"{question}\n{INSTRUCTION}"
    messages = [{"role": "user", "content": FEWSHOT + f"Q: {full_question}\nA:"}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [7]:
def generate_greedy(prompt, max_new_tokens=768):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,       # greedy
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text

In [8]:
import re

def extract_answer(text):
    text = text.replace(",", "").replace("**", "")
    match = re.search(r"[Tt]he answer is\s*\$?(-?\d+\.?\d*)", text)
    if match:
        return match.group(1).rstrip(".")
    else:
        matches = re.findall(r"-?\d+\.?\d*", text)

    return matches[-1] if matches else None

def values_match(pred, gt):
    if pred is None:
        return False
    try:
        return float(pred) == float(gt)
    except ValueError:
        return False

def extract_gt(answer_field):
    return answer_field.split("####")[-1].strip().replace(",", "")

In [ ]:
n = 50
correct = 0
for i in range(n):
    print(i)
    ex = test_set[i]
    prompt = build_prompt(ex["question"])
    out = generate_greedy(prompt)
    #print(out)
    pred = extract_answer(out)
    #print(pred)
    gt = extract_gt(ex["answer"])
    #print(gt)
    correct += (pred == gt)

print(f"Greedy accuracy: {correct/n:.2%}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
Greedy accuracy: 74.00%


In [9]:
def generate_samples(prompt, temperature, n_samples=10, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        num_return_sequences=n_samples,
        pad_token_id=tokenizer.eos_token_id
    )
    return [tokenizer.decode(output[inputs["input_ids"].shape[1]:], skip_special_tokens=True) for output in outputs]

In [11]:
import os, json

CHECKPOINT_PATH = "/content/drive/MyDrive/gridworld-sc/sc_results.json"

# Ensure the directory exists
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

print(f"max_new_tokens = {max_new_tokens}")
print(f"temperatures = {temperatures}")
print(f"n_samples = {n_samples}")
print(f"questions = {len(test_set)}")
input("Press enter to confirm and start the run...")  # forces a manual check

# Resume logic: load existing results if present
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        results = json.load(f)
    done_indices = {r["idx"] for r in results}
    print(f"Resuming — {len(done_indices)} questions already done")
else:
    results = []
    done_indices = set()

for i, ex in enumerate(test_set.select(range(200))):
    if i in done_indices:
        continue  # skip work already saved
    prompt = build_prompt(ex["question"])
    for temp in [0.5, 0.7, 1.0]:
        samples = generate_samples(prompt, temperature=temp, n_samples=10)
        results.append({"idx": i, "temp": temp, "samples": samples, "gt": extract_gt(ex["answer"])})
    if i % 10 == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(results, f)
        print(f"checkpointed at {i}")

with open(CHECKPOINT_PATH, "w") as f:
    json.dump(results, f)

checkpointed at 0
checkpointed at 10
checkpointed at 20
checkpointed at 30
checkpointed at 40
checkpointed at 50
checkpointed at 60
checkpointed at 70
checkpointed at 80
checkpointed at 90
checkpointed at 100
checkpointed at 110
checkpointed at 120
checkpointed at 130
checkpointed at 140
checkpointed at 150
checkpointed at 160
checkpointed at 170
checkpointed at 180
checkpointed at 190


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
